In [0]:
# CELL 1
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml import Pipeline
from pyspark.sql import functions as F

df = spark.read.table("sarkarimitracatalog.sarkaridatabase.schemes_structured")
feature_cols = ["age_min", "age_max", "income_limit_inr", "extraction_confidence"]
df_feat = df.select(["slug", "scheme_name"] + feature_cols).fillna(0)

print(f"✅ Loaded {df_feat.count()} schemes for clustering")

✅ Loaded 3399 schemes for clustering


In [0]:
# CELL 2 — KMeans clustering
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
scaler    = StandardScaler(inputCol="features", outputCol="scaled_features",
                            withMean=True, withStd=True)
kmeans    = KMeans(k=8, seed=42,
                   featuresCol="scaled_features",
                   predictionCol="cluster_id")

pipeline     = Pipeline(stages=[assembler, scaler, kmeans])
model        = pipeline.fit(df_feat)
df_clustered = model.transform(df_feat) \
    .select("slug", "scheme_name", "cluster_id")

print("✅ Cluster distribution:")
display(df_clustered.groupBy("cluster_id").count().orderBy("cluster_id"))

✅ Cluster distribution:


cluster_id,count
0,1938
1,135
2,1
3,4
4,922
5,12
6,195
7,192


In [0]:
# CELL 3 — Write
df_clustered.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sarkarimitracatalog.sarkaridatabase.scheme_clusters")

print("✅ Written: sarkarimitracatalog.sarkaridatabase.scheme_clusters")

✅ Written: sarkarimitracatalog.sarkaridatabase.scheme_clusters
